In [12]:
import pandas as pd
import numpy as np
import re 

# Load Data

In [2]:
df = pd.read_csv("./spam.csv", encoding="latin-1")
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
df.Category.value_counts()

Category
ham     4825
spam     747
Name: count, dtype: int64

In [4]:
df["Category"] = df["Category"].astype(str).str.strip().str.lower()

In [6]:
df['target'] = df['Category'].map({"ham":0, "spam":1})

In [8]:
df = df.dropna(subset=["Message"]).reset_index(drop=True)

In [9]:
df

,Category,Message,target
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1
5568,ham,Will Ã¼ b going to esplanade fr home?,0
5569,ham,"Pity, * was in mood for that. So...any other s...",0
5570,ham,The guy did some bitching but I acted like i'd...,0


# Text cleaning

In [10]:
# making all the text lowercase
df['Message'] = df['Message'].str.lower()

In [16]:
df["Message"] = df["Message"].apply(lambda text: re.sub(r"http\S+|www\.\S+", " ", str(text)))

In [17]:
df["Message"] = df["Message"].apply(lambda text: re.sub(r"[^a-z0-9\s]", " ", str(text)))

In [18]:
df["Message"] = df["Message"].apply(lambda text: " ".join(str(text).split()))

In [19]:
df.head(3)

,Category,Message,target
0,ham,go until jurong point crazy available only in ...,0
1,ham,ok lar joking wif u oni,0
2,spam,free entry in 2 a wkly comp to win fa cup fina...,1


In [20]:
# text cleaning in one fuction 
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = " ".join(text.split())
    return text

In [21]:
# apply the clean_text function to the Message column 
df["Message"] = df["Message"].apply(clean_text)

In [23]:
df

,Category,Message,target
0,ham,go until jurong point crazy available only in ...,0
1,ham,ok lar joking wif u oni,0
2,spam,free entry in a wkly comp to win fa cup final ...,1
3,ham,u dun say so early hor u c already then say,0
4,ham,nah i don t think he goes to usf he lives arou...,0
...,...,...,...
5567,spam,this is the nd time we have tried contact u u ...,1
5568,ham,will b going to esplanade fr home,0
5569,ham,pity was in mood for that so any other suggest...,0
5570,ham,the guy did some bitching but i acted like i d...,0


# Split Data 

In [24]:
from sklearn.model_selection import train_test_split

x = df["Message"]
y = df["target"]

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [26]:
X_test.shape

(1115,)

In [27]:
X_train.shape

(4457,)

# Bag Of Words using Pipeline + Model Building 

In [32]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB

In [33]:
pipeline = Pipeline([
    ("vectorizer", CountVectorizer(
       lowercase=True,
       stop_words="english",
       ngram_range=(1,2),
       min_df=5,
    )),
    ("model", MultinomialNB())
])

## Model Train 

In [34]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('vectorizer',
                 CountVectorizer(min_df=5, ngram_range=(1, 2),
                                 stop_words='english')),
                ('model', MultinomialNB())])

In [35]:
y_pred = pipeline.predict(X_test)

In [36]:
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(1115,))

In [37]:
from sklearn.metrics import classification_report, accuracy_score

In [38]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.97847533632287


In [39]:
print("\nClassification Report:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=["ham","spam"]
))


Classification Report:

              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       966
        spam       0.94      0.90      0.92       149

    accuracy                           0.98      1115
   macro avg       0.96      0.95      0.95      1115
weighted avg       0.98      0.98      0.98      1115



## Test on new message

In [40]:
new_messages = [
    "Hi, can we meet tomorrow for lunch?",
    "You have won a free prize! Call now to claim your reward.",
    "Please check the attached invoice."
]

In [41]:
predictions = pipeline.predict(new_messages)

In [42]:
for msg, pred in zip(new_messages, predictions):
    label = "spam" if pred == 1 else "ham"
    print(f"Message: {msg}")
    print("Predication:", label)
    print("-"*50)
    

Message: Hi, can we meet tomorrow for lunch?
Predication: ham
--------------------------------------------------
Message: You have won a free prize! Call now to claim your reward.
Predication: spam
--------------------------------------------------
Message: Please check the attached invoice.
Predication: ham
--------------------------------------------------
